### Master of Applied Artificial Intelligence

**Course: TC5035 - Proyecto Integrador**

<img src="https://github.com/Medicenchapin/Proyecto-Integrador/blob/main/assets/logo.png?raw=1" alt="Image Alt Text" width="500"/>


**Other models**

Tutor: Dr. Horario Martinez Alfaro


Team members:
* Ignacio Jose Aguilar Garcia - A00819762
* Alejandro Calderon Aguilar - A01795353
* Ricardo Mar Cupido - A01795394

# Imports


In [5]:
import sys
sys.path.append('../')
import os
import pandas as pd
import importlib, scripts.helpers as hp
importlib.reload(hp)
from scripts.helpers import Helpers

# import df

In [8]:
df = pd.read_parquet("../data/campaign_candidates_final.parquet")
df.tail()

,state_name,previous_classification,previous_calls,client_age,network_age_years,banking,arpu_90_days,minutes_in,validity_average,average_performance,...,plan_postpaid,sn_banking,digital_index_mean,connected_days,charged_days,apps_days,music_gb,proba,sample_idx,drivers
18298,GUATEMALA,NEW CLIENT,0,22.0,3.16,1,103.42,38.98,2.00,0.70,...,0.50,0.75,0.89,91,91,0,0.334,0.694028,182992,"[{'feature': 'client_age', 'impact': 0.4018475..."
18299,GUATEMALA,NEW CLIENT,0,39.0,0.22,1,101.54,21.88,2.60,0.51,...,0.79,0.86,0.32,75,56,3,0.000,0.756829,182993,"[{'feature': 'plan_postpaid', 'impact': 0.4847..."
18300,JUTIAPA,NEW CLIENT,0,40.0,0.24,0,92.20,12.46,14.00,0.73,...,0.43,0.43,0.00,55,50,2,0.000,0.628511,182998,"[{'feature': 'plan_postpaid', 'impact': 0.5019..."
18301,PETEN,NOT EFFECTIVE,2,NaN,10.57,1,100.21,42.05,7.38,0.55,...,0.27,0.45,0.56,91,67,13,0.481,0.675360,183004,"[{'feature': 'plan_postpaid', 'impact': 0.2827..."
18302,GUATEMALA,NEW CLIENT,0,NaN,0.23,0,109.61,79.16,5.38,0.73,...,0.56,0.69,0.00,79,69,6,0.000,0.805487,183012,"[{'feature': 'plan_postpaid', 'impact': 0.5441..."


# Instance of helpers

In [6]:
helpers = Helpers(df=df)

# Global context (prompt)

In [7]:
global_prompt = helpers.build_global_system_prompt_es()
print(global_prompt)

Eres un asistente analítico para una empresa de telecomunicaciones. Tu función es ayudar a interpretar los principales drivers (valores SHAP) del modelo a nivel global y por cliente, en términos de negocio.

            Resumen Global de Drivers SHAP
            Estas son las variables globalmente más influyentes (TOP 10) y su significado de negocio:
            - plan_postpaid: Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- music_gb: Cantidad de datos móviles (en GB) usados para música. Un valor cero representa oportunidad para ofertas de 'música sin consumo de datos'.
- client_age: Edad del cliente en años. Evita sesgos demográficos; úsala solo para ajustar el tono de comunicación si es necesario.
- contacts: Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- network_age_years: Años desde que el cliente se unió 

# select 5 customers

In [26]:
sample = df.sample(5, random_state=42)
customers_idx = [s for s in sample['sample_idx']]
customer_prompts = []
for idx, i in enumerate(customers_idx, 0):
    row = sample.iloc[idx]
    customer_prompt = helpers.build_customer_prompt_summary(
        row=row,
        driver_list=row["drivers"]
    )
    customer_prompts.append(customer_prompt)

In [27]:
for customer_prompt in customer_prompts:
    print(customer_prompt)

Eres un analista de campañas de telecomunicaciones prepago.

        Analiza los factores más influyentes en la probabilidad de compra para un cliente individual, basándote en valores SHAP.
        El modelo predijo una probabilidad de aceptación del **65.7%**.

        A continuación se listan los principales *drivers* (variables) que explican esta predicción,
        ordenados por relevancia:

        - **plan_postpaid** (positivo (favorece contacto)): valor = 1.35. Indica si el cliente tiene un plan pospago (1 = sí, 0 = no). Los clientes pospago valoran la experiencia premium, estabilidad y confiabilidad.
- **contacts** (positivo (favorece contacto)): valor = 1.35. Total de contactos previos con la empresa. Si es alto y el SHAP es negativo, sugiere reducir fricción y simplificar procesos.
- **start_using_months** (negativo (revisar antes de contactar)): valor = 1.00. Meses desde que el cliente comenzó a usar el servicio. Usuarios con más tiempo pueden reengancharse con ofertas de le

# Ollama - Local LLM

## version 1

In [ ]:
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "cas/nous-hermes-2-mistral-7b-dpo")

## Version cas/nous-hermes-2-mistral-7b-dpo

A fine-tuned Mistral-7B instruction model using DPO (Direct Preference Optimization). It’s based on OpenHermes 2.5 and shows broad benchmark gains vs its base. 
(https://huggingface.co/NousResearch/Nous-Hermes-2-Mistral-7B-DPO?utm_source=chatgpt.com)

Ollama publishes a ready-to-run quantized build (GGUF) under this name. Typical quant is Q4_K_M (~4.4 GB). Model card metadata shows 7.24B params; template uses system/user turns.


Advantages:

* ✅ Multilingual (ES/EN) assistant tasks, analytics summaries, customer-service style outputs.
* ✅ CPU-only Mac or small GPU server with quantized weights via Ollama.

# OpenAI - API LLM

## version x

## version y